# MazeRunner · repeated laps on one map (beyond the training horizon)

The second MazeRunner beyond-horizon evaluation (docs/EXPERIMENTS.md, long-running evaluations). Every confirmation map is replayed by the frozen 8M endpoints lap after lap for 4,000 charged calls: a lap is one native episode of the trained task on the map's own seed — the same maze, the same ordered goals and the same hidden action permutation, the agent back at the start cell, the native 500-step timer — and a reset-only call separates consecutive laps while the carried memory continues across it. The packet is the trained one field for field, so only one thing changes against training: how long the memory has to last. At 4,000 calls the full-history Transformer attends over eight times the records it was trained on and Memo holds eight times its trained summary blocks, while RSM's carried state is the same forty slots.

A carrier that keeps what it learned about the walls and the action map should need fewer calls per lap after the first and reach more goals per window; one that loses it falls back towards its first-lap rate. Panel (a) is the rate per 500-call window, (b) the cumulative goals, (c) what the memory carried across lap boundaries adds — the same policy with the memory kept minus with it wiped at every boundary (lap-cleared), paired per map and seed — and (d) the last window against the peak live state an actor holds within the task, the allocation for the trained horizon as a hollow marker where it differs.

This axis and the larger-maze axis (appendix A5) are different extrapolations: more calls on the same map against a larger map under an area-scaled budget, so a cell can hold on one and fall on the other; they are never read as one result. Uncertainty is the joint seed/map bootstrap (95 %, conditional on the three trained seeds). The memoryless control is not drawn (nothing to carry) and is read in the tables. Cells without complete panels are named in the legend, never drawn from another roster.


In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import json
import textwrap
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from figures_common import (
    ABLATION_CONDITION,
    DRAWN_CELLS,
    ENVIRONMENTS,
    FULL_WIDTH,
    INK_FAINT,
    INK_MUTED,
    METHOD_CONDITION,
    PANEL,
    SEEDS,
    BuiltFigure,
    CellStyle,
    band,
    cell_state_rows,
    composite_legend,
    laps_panel,
    legend_handles,
    paper_style,
    pending_handles,
    standalone,
    style_axes,
)
from matplotlib.lines import Line2D

from reasoned_icrl.analysis.statistics import _bootstrap, _interval
from reasoned_icrl.experiments.horizon import WINDOW, window_goal_matrix

LAPS_BUDGET = 4000
"""The repeated-laps budget in charged calls:
eight times the trained 500-call episode. The 1,000- and 2,000-call readings
are its first two and four windows (the rollout is prefix-consistent)."""
DELTA = 1.0
"""The practical margin: one goal per 500-call window."""
REFERENCE_WINDOW = 2
"""The last window is compared with calls 501-1,000, the first window after
a route can have been learned (window 1 holds the exploring first lap)."""
SHORT = {
    "raw_summary": "RSM-O",
    "raw_segment": "w/o memory",
    "raw_summary_residual": "RSM-R",
    "full_context": "full history",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo fixed",
}
LAPS_CELLS = tuple(c for c in DRAWN_CELLS if c.condition != "raw_segment")
"""The memory carriers, as in every beyond-horizon figure: the RSM family, the full history, the GRU, Memo and Memo
fixed. The memoryless control is read in the tables."""
ALLOCATION_TOLERANCE = 0.02
ALLOCATION_NOTE = "hollow marker: allocated for the trained horizon"
TWO_AXES_NOTE = (
    "Repeated laps extend only how long the memory must last (same map, same "
    "task); the larger-maze axis (appendix) is a different extrapolation."
)


@dataclass(frozen=True)
class LapsOptions:
    seeds: Sequence[int] = SEEDS
    samples: int = 2000
    resampling_seed: int = 0
    split: str = "confirmation"
    """The paper's roster; ``development`` draws the pre-read (provisional)."""
    budget: int = LAPS_BUDGET
    cells: Sequence[CellStyle] = LAPS_CELLS
    control: str = "raw_segment"
    method: str = METHOD_CONDITION
    state_accounting: str = "peak"
    name: str = "figure_mazerunner_repeated_laps"
    figsize: tuple[float, float] = (FULL_WIDTH, 4.1)
    panel_size: tuple[float, float] = PANEL
    caption: bool = True


# ---------------------------------------------------------------- estimator
# Every goal of every lap is binned by the charged call that reached it
# (``goal_steps``), the partial lap's goals included: goals per 500-call
# window per (map, rollout) unit, one layer per seed.


def load_cube(layout, condition, history, contract, options, sources):
    """``[seeds, units, windows]`` goals per window, or ``None`` when any seed
    lacks a complete laps panel."""
    units = [
        (int(task), int(rollout))
        for task in contract.roster(options.split)
        for rollout in contract.evaluation.rollout_seeds
    ]
    layers, staged = [], []
    for seed in options.seeds:
        events = laps_panel(
            layout,
            condition,
            seed,
            options.split,
            history,
            options.budget,
            contract=contract,
            sources=staged,
        )
        if events is None:
            return None
        layers.append(
            window_goal_matrix(events, units=units, outer_length=options.budget)
        )
    sources.extend(staged)
    return np.stack(layers)


def lap_calls(layout, condition, contract, options):
    """Calls per finished lap by lap index (1-based), pooled over seeds and maps."""
    by_lap = {}
    for seed in options.seeds:
        events = laps_panel(
            layout,
            condition,
            seed,
            options.split,
            "retained",
            options.budget,
            contract=contract,
        )
        for e in events or ():
            if e.complete:
                by_lap.setdefault(int(e.event_index), []).append(int(e.step))
    return by_lap


def estimate(matrix, *, samples, rng, seeds):
    """Mean of a ``[seeds, units]`` matrix, its joint bootstrap interval, the seed means."""
    value = float(matrix.mean())
    lower, upper = _interval(_bootstrap(matrix, samples=samples, rng=rng), value, 0.95)
    return {
        "estimate": value,
        "lower": lower,
        "upper": upper,
        **{f"seed_{s}": float(matrix[i].mean()) for i, s in enumerate(seeds)},
    }


def window_rows(cubes, options, rng):
    rows = []
    for condition, cube in cubes.items():
        if cube is None:
            continue
        running = cube.cumsum(axis=2)
        for w in range(cube.shape[2]):
            rate = estimate(
                cube[:, :, w], samples=options.samples, rng=rng, seeds=options.seeds
            )
            total = estimate(
                running[:, :, w], samples=options.samples, rng=rng, seeds=options.seeds
            )
            rows.append(
                {
                    "condition": condition,
                    "window": w + 1,
                    "first_call": w * WINDOW + 1,
                    "last_call": (w + 1) * WINDOW,
                    "centre": (w + 0.5) * WINDOW,
                    **rate,
                    **{f"cumulative_{k}": v for k, v in total.items()},
                }
            )
    return rows


def gain_rows(retained, cleared, options, rng):
    """Goals per window retained minus lap-cleared, paired per unit: what the
    memory carried across lap boundaries is worth, window by window."""
    rows = []
    for condition, cube in retained.items():
        other = cleared.get(condition)
        if cube is None or other is None:
            continue
        diff = cube - other
        for w in range(diff.shape[2]):
            rows.append(
                {
                    "condition": condition,
                    "window": w + 1,
                    "centre": (w + 0.5) * WINDOW,
                    **estimate(
                        diff[:, :, w],
                        samples=options.samples,
                        rng=rng,
                        seeds=options.seeds,
                    ),
                }
            )
    return rows


def readings(retained, cleared, options, rng):
    """The method's last window against calls 501-1,000 and against every other
    memory, and each memory's change under lap-cleared, paired per map and seed."""
    method = options.method
    rows = []

    def add(name, matrix, *, test):
        e = estimate(matrix, samples=options.samples, rng=rng, seeds=options.seeds)
        per = [e[f"seed_{s}"] for s in options.seeds]
        rows.append(
            {
                "reading": name,
                "method": method,
                **e,
                "holds": bool(test(e, per)),
            }
        )

    me = retained.get(method)
    if me is not None:
        last = me.shape[2] - 1
        add(
            f"{SHORT[method]} last window - calls 501-1,000",
            me[:, :, last] - me[:, :, REFERENCE_WINDOW - 1],
            test=lambda e, per: all(abs(v) <= DELTA for v in per),
        )
        for condition in (
            "full_context",
            "memo",
            "memo_fixed",
            "full_gru",
            options.control,
        ):
            other = retained.get(condition)
            if other is None or condition == method:
                continue
            add(
                f"{SHORT[method]} - {SHORT[condition]}, last window",
                me[:, :, last] - other[:, :, last],
                test=lambda e, per: e["lower"] > 0 and all(v >= DELTA for v in per),
            )
    for condition, cube in cleared.items():
        if cube is None:
            continue
        last = cube.shape[2] - 1
        add(
            f"{SHORT[condition]} lap-cleared: last window - first window",
            cube[:, :, last] - cube[:, :, 0],
            test=lambda e, per: all(abs(v) <= DELTA for v in per),
        )
    return rows


def prefix_status(layout, condition, options):
    """The acceptance check each laps panel wrote (lap 1 = the plain episode)."""
    statuses = []
    for seed in options.seeds:
        path = (
            layout.run_directory("mazerunner", condition, seed)
            / "eval"
            / f"{options.split}-retained-endpoint-calls{options.budget}"
            / "prefix_check.json"
        )
        statuses.append(
            json.loads(path.read_text())["status"] if path.is_file() else "absent"
        )
    return statuses


def compute_laps(layout, options, sources):
    contract = layout.contract("mazerunner")
    conditions = [c.condition for c in options.cells] + [options.control]
    retained = {
        c: load_cube(layout, c, "retained", contract, options, sources)
        for c in conditions
    }
    cleared = {
        c: load_cube(layout, c, "attempt-cleared", contract, options, sources)
        for c in conditions
    }
    rng = np.random.default_rng(options.resampling_seed)
    present = [c for c in options.cells if retained.get(c.condition) is not None]
    missing = [c for c in options.cells if retained.get(c.condition) is None]
    states = cell_state_rows(
        layout,
        "mazerunner",
        [c.condition for c in present],
        options.budget,
        contract=contract,
        seeds=options.seeds,
        sources=sources,
        accounting=options.state_accounting,
        records=options.budget + 1,
    )
    calls = {
        c: lap_calls(layout, c, contract, options)
        for c in conditions
        if retained.get(c) is not None
    }
    units = len(contract.roster(options.split)) * len(contract.evaluation.rollout_seeds)
    return {
        "present": present,
        "missing": missing,
        "windows": window_rows(retained, options, rng),
        "gains": gain_rows(retained, cleared, options, rng),
        "readings": readings(retained, cleared, options, rng),
        "states": states,
        "calls": calls,
        "units": units,
        "checks": {
            c: prefix_status(layout, c, options)
            for c in conditions
            if retained.get(c) is not None
        },
        "cleared_present": [
            c for c in options.cells if cleared.get(c.condition) is not None
        ],
    }


# ------------------------------------------------------------------ drawing


def _calls_axis(ax, budget):
    step = 1000 if budget >= 3000 else 500
    ticks = list(range(0, budget + 1, step))
    ax.set_xlim(0, budget)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t:,}" for t in ticks])
    ax.axvspan(WINDOW, budget, color="#f4f4f4", zorder=0)
    ax.axvline(WINDOW, color=INK_FAINT, linewidth=0.7, linestyle=(0, (2, 2)), zorder=1)


def draw_goals_per_window(ax, data, options):
    """(a) goals reached per 500-call window: the trained episode shaded off,
    the extension grey; seeds as thin lines."""
    top = 5.0
    for cell in data["present"]:
        mine = [r for r in data["windows"] if r["condition"] == cell.condition]
        x = [r["centre"] for r in mine]
        for seed in options.seeds:
            ax.plot(
                x,
                [r[f"seed_{seed}"] for r in mine],
                color=cell.color,
                linewidth=0.45,
                alpha=0.35,
                linestyle=cell.linestyle,
                zorder=cell.zorder + 5,
            )
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.4))
        top = max(top, 1.05 * max(r["upper"] for r in mine))
    _calls_axis(ax, options.budget)
    ax.text(
        WINDOW + 0.01 * options.budget,
        0.97,
        "trained episode",
        transform=ax.get_xaxis_transform(),
        color=INK_MUTED,
        fontsize=8.0,
        va="top",
        ha="left",
    )
    ax.set_ylim(0, top)
    ax.set_xlabel("Charged calls on one map")
    ax.set_ylabel("Goals per 500-call window")
    ax.set_title("(a) goals per window")
    style_axes(ax)


def draw_cumulative_goals(ax, data, options):
    """(b) cumulative goals reached against charged calls."""
    top = 10.0
    for cell in data["present"]:
        mine = [r for r in data["windows"] if r["condition"] == cell.condition]
        x = [0.0, *(r["last_call"] for r in mine)]
        band(
            ax,
            x,
            [0.0, *(r["cumulative_lower"] for r in mine)],
            [0.0, *(r["cumulative_upper"] for r in mine)],
            cell,
        )
        ax.plot(x, [0.0, *(r["cumulative_estimate"] for r in mine)], **cell.line())
        top = max(top, 1.05 * max(r["cumulative_upper"] for r in mine))
    _calls_axis(ax, options.budget)
    ax.set_ylim(0, top)
    ax.set_xlabel("Charged calls on one map")
    ax.set_ylabel("Cumulative goals reached")
    ax.set_title("(b) cumulative goals")
    style_axes(ax)


def draw_memory_gain(ax, data, options):
    """(c) goals per window with the memory kept across lap boundaries minus
    with it wiped at every boundary (lap-cleared), paired per map and seed."""
    shown = [
        c
        for c in data["present"]
        if any(r["condition"] == c.condition for r in data["gains"])
    ]
    low, high = -1.0, 1.0
    for cell in shown:
        mine = [r for r in data["gains"] if r["condition"] == cell.condition]
        x = [r["centre"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.4))
        low = min(low, min(r["lower"] for r in mine))
        high = max(high, max(r["upper"] for r in mine))
    ax.axhline(0.0, color=INK_FAINT, linewidth=0.7, zorder=1)
    _calls_axis(ax, options.budget)
    pad = 0.08 * (high - low)
    ax.set_ylim(low - pad, high + pad)
    ax.set_xlabel("Charged calls on one map")
    ax.set_ylabel("Kept - wiped at each lap, goals/window")
    ax.set_title("(c) memory across laps")
    style_axes(ax)


def _state_axis(ax, values):
    ax.set_xscale("log")
    low = min(values) if values else 1e4
    high = max(values) if values else 1e7
    ax.set_xlim(low / 4, high * 4)
    labels = {
        1e3: "1 KB",
        1e4: "10 KB",
        1e5: "100 KB",
        1e6: "1 MB",
        1e7: "10 MB",
        1e8: "100 MB",
    }
    ticks = [t for t in labels if low / 4 <= t <= high * 4]
    if len(ticks) > 3:
        ticks = ticks[::2]
    ax.set_xticks(ticks)
    ax.set_xticklabels([labels[t] for t in ticks])
    ax.minorticks_off()


def draw_last_window_against_state(ax, data, options, letter="d"):
    """(d) goals in the last window against the peak live state an actor holds
    within the 4,000-call task; allocation as a hollow marker where it differs."""
    last = max((r["window"] for r in data["windows"]), default=0)
    points = {r["condition"]: r for r in data["windows"] if r["window"] == last}
    states = {r["condition"]: r["bytes"] for r in data["states"]}
    reserved = {r["condition"]: r["allocated_bytes"] for r in data["states"]}
    xs = []
    for cell in data["present"]:
        if cell.condition not in states or cell.condition not in points:
            continue
        row, x = points[cell.condition], states[cell.condition]
        xs.append(x)
        for seed in options.seeds:
            ax.plot(
                x,
                row[f"seed_{seed}"],
                marker=cell.marker,
                markersize=3.0,
                color=cell.color,
                alpha=0.45,
                linestyle="none",
                markerfacecolor=cell.marker_face,
                markeredgewidth=0.5,
                zorder=cell.zorder + 5,
            )
        ax.errorbar(
            x,
            row["estimate"],
            yerr=[[row["estimate"] - row["lower"]], [row["upper"] - row["estimate"]]],
            color=cell.color,
            elinewidth=0.7,
            capsize=0,
            marker=cell.marker,
            markersize=5.0,
            markerfacecolor=cell.marker_face,
            markeredgewidth=0.8,
            linestyle="none",
            zorder=cell.zorder + 10,
        )
        r = reserved.get(cell.condition)
        if r is not None and abs(r - x) > ALLOCATION_TOLERANCE * max(r, x):
            xs.append(r)
            ax.plot(
                [min(x, r), max(x, r)],
                [row["estimate"]] * 2,
                color=cell.color,
                linewidth=0.6,
                alpha=0.5,
                zorder=cell.zorder + 6,
            )
            ax.plot(
                r,
                row["estimate"],
                marker=cell.marker,
                markersize=4.4,
                color=cell.color,
                markerfacecolor="white",
                markeredgewidth=0.9,
                linestyle="none",
                zorder=cell.zorder + 9,
            )
    _state_axis(ax, xs)
    ax.set_ylim(bottom=0)
    first = (last - 1) * WINDOW + 1
    ax.set_ylabel(
        f"Goals per 500 calls, calls {first:,} to {last * WINDOW:,}"
        if last
        else "Goals per 500 calls"
    )
    ax.set_xlabel(
        "Peak live state per actor"
        if options.state_accounting == "peak"
        else "Average live state per actor"
    )
    ax.set_title(f"({letter}) last window vs state")
    style_axes(ax, grid_axis="both")


def caption(data, options):
    readings = data["readings"]
    if not readings:
        return ""

    def fmt(r):
        return f"{r['estimate']:+.1f} [{r['lower']:+.1f}, {r['upper']:+.1f}]"

    parts = []
    for r in readings:
        name = r["reading"]
        if "lap-cleared" in name:
            continue
        parts.append(f"{name.replace(', last window', '')} {fmt(r)}")
    last = max(r["window"] for r in data["windows"])
    head = (
        f"Goals per 500-call window on one map replayed in laps for {options.budget:,} "
        f"charged calls ({data['units']} maps, {len(options.seeds)} seeds, joint "
        "seed/map 95 % intervals; the trained episode is 500 calls). Last window, "
        f"calls {(last - 1) * WINDOW + 1:,} to {last * WINDOW:,}"
        + ": "
        + "; ".join(parts)
        + "."
    )
    cleared = [r for r in readings if "lap-cleared" in r["reading"]]
    if cleared:
        head += (
            " Lap-cleared, last minus first window: "
            + "; ".join(
                f"{r['reading'].split(' lap-cleared', 1)[0]} {r['estimate']:+.1f}"
                for r in cleared
            )
            + "."
        )
    return "\n".join(textwrap.wrap(head + " " + TWO_AXES_NOTE, 150))


def build_laps(layout, options=None):
    options = options or LapsOptions()
    sources = []
    data = compute_laps(layout, options, sources)
    handles = legend_handles(data["present"])
    handles.append(
        Line2D(
            [],
            [],
            label=ALLOCATION_NOTE,
            linestyle="none",
            marker="o",
            markersize=4.4,
            color=INK_MUTED,
            markerfacecolor="white",
            markeredgewidth=0.9,
        )
    )
    handles += pending_handles(
        {"MazeRunner laps": data["missing"]},
        options.cells,
        reason="pending (no complete panels)",
    )
    title = (
        f"{ENVIRONMENTS['mazerunner'].title} · one map replayed in laps for {options.budget:,} charged calls, frozen policies"
        + (
            ""
            if options.split == "confirmation"
            else f" ({options.split} roster, provisional)"
        )
    )
    with paper_style():
        fig = plt.figure(figsize=options.figsize, layout="constrained")
        # The lap-cleared panel is drawn only once its panels exist: nothing
        # missing is drawn as an empty chart.
        gains = bool(data["gains"])
        axes = fig.subplots(1, 4 if gains else 3)
        draw_goals_per_window(axes[0], data, options)
        draw_cumulative_goals(axes[1], data, options)
        if gains:
            draw_memory_gain(axes[2], data, options)
        draw_last_window_against_state(
            axes[-1], data, options, letter="d" if gains else "c"
        )
        composite_legend(fig, handles, ncol=4, title=title)
        text = caption(data, options) if options.caption else ""
        if text:
            fig.text(
                0.5, -0.01, text, ha="center", va="top", fontsize=8.0, color=INK_MUTED
            )
    panels = {
        "goals_per_window": standalone(
            lambda ax: draw_goals_per_window(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
        "cumulative_goals": standalone(
            lambda ax: draw_cumulative_goals(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
        "last_window_against_state": standalone(
            lambda ax: draw_last_window_against_state(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
    }
    if data["gains"]:
        panels["memory_gain"] = standalone(
            lambda ax: draw_memory_gain(ax, data, options),
            size=options.panel_size,
            handles=handles,
        )
    lap_table = [
        {
            "condition": c,
            "lap": lap,
            "finished": len(v),
            "calls_mean": float(np.mean(v)),
        }
        for c, by in data["calls"].items()
        for lap, v in sorted(by.items())
        if lap <= 20
    ]
    tables = {
        "windows": data["windows"],
        "memory_gain": data["gains"],
        "readings": data["readings"],
        "state": data["states"],
        "calls_per_lap": lap_table,
    }
    notes = {
        "split": options.split,
        "budget": options.budget,
        "present": [c.condition for c in data["present"]],
        "missing": [c.condition for c in data["missing"]],
        "lap_cleared_present": [c.condition for c in data["cleared_present"]],
        "prefix_checks": data["checks"],
        "units": data["units"],
        "seeds": list(options.seeds),
        "samples": options.samples,
        "delta": DELTA,
        "checkpoint": "policy_epoch_999 (collection endpoint)",
        "interval": "95 % joint seed/map bootstrap, conditional on the trained seeds",
        "state": {r["condition"]: r["method"] for r in data["states"]},
        "axes": TWO_AXES_NOTE,
    }
    return BuiltFigure(options.name, fig, tables, notes, sources, panels)


# ------------------------------------------------------- map change (M1, M2)
# The repeated-laps task with a new 15x15 maze and goal sequence every 1,000
# calls (the action map is kept and the memory carries on). Goals are attributed to the map their lap ran on.

MAP_PERIOD = 1000
"""Calls between map changes (four maps in the 4,000-call task)."""


@dataclass(frozen=True)
class MapChangeOptions:
    seeds: Sequence[int] = SEEDS
    samples: int = 2000
    resampling_seed: int = 0
    split: str = "confirmation"
    budget: int = LAPS_BUDGET
    period: int = MAP_PERIOD
    cells: Sequence[CellStyle] = LAPS_CELLS
    method: str = METHOD_CONDITION
    name: str = "figure_mazerunner_map_change"
    figsize: tuple[float, float] = (FULL_WIDTH, 3.9)
    panel_size: tuple[float, float] = PANEL
    caption: bool = True


def map_cube(layout, condition, contract, options, sources):
    """Per seed and roster unit: goals per 500 calls on each map, the calls
    the first finished lap on each map took, and goals per 500-call window;
    ``None`` when any seed lacks a complete panel."""
    units = [
        (int(task), int(rollout))
        for task in contract.roster(options.split)
        for rollout in contract.evaluation.rollout_seeds
    ]
    maps = options.budget // options.period
    rates, first, windows, staged = [], [], [], []
    for seed in options.seeds:
        events = laps_panel(
            layout,
            condition,
            seed,
            options.split,
            "retained",
            options.budget,
            contract=contract,
            sources=staged,
            layout_period=options.period,
        )
        if events is None:
            return None
        index = {u: i for i, u in enumerate(units)}
        goals = np.zeros((len(units), maps))
        calls = np.zeros((len(units), maps))
        lap1 = np.full((len(units), maps), np.nan)
        for e in events:
            row = index[(int(e.task_id), int(e.rollout_seed))]
            k = int(e.layout_index or 0)
            goals[row, k] += e.numerator
            calls[row, k] += e.step
            if e.complete and np.isnan(lap1[row, k]):
                lap1[row, k] = e.step
        rates.append(500.0 * goals / np.maximum(calls, 1.0))
        first.append(lap1)
        windows.append(
            window_goal_matrix(events, units=units, outer_length=options.budget)
        )
    sources.extend(staged)
    return np.stack(rates), np.stack(first), np.stack(windows)


def compute_map_change(layout, options, sources):
    contract = layout.contract("mazerunner")
    rng = np.random.default_rng(options.resampling_seed)
    conditions = [c.condition for c in options.cells] + ["raw_segment"]
    cubes = {c: map_cube(layout, c, contract, options, sources) for c in conditions}
    present = [c for c in options.cells if cubes.get(c.condition) is not None]
    missing = [c for c in options.cells if cubes.get(c.condition) is None]
    per_map, first_lap, windows = [], [], []
    for condition, cube in cubes.items():
        if cube is None:
            continue
        rate, lap1, win = cube
        for k in range(rate.shape[2]):
            per_map.append(
                {
                    "condition": condition,
                    "map": k + 1,
                    **estimate(
                        rate[:, :, k],
                        samples=options.samples,
                        rng=rng,
                        seeds=options.seeds,
                    ),
                }
            )
            finite = np.where(np.isnan(lap1[:, :, k]), options.budget, lap1[:, :, k])
            first_lap.append(
                {
                    "condition": condition,
                    "map": k + 1,
                    "censored_at_budget": int(np.isnan(lap1[:, :, k]).sum()),
                    **estimate(
                        finite, samples=options.samples, rng=rng, seeds=options.seeds
                    ),
                }
            )
        for w in range(win.shape[2]):
            windows.append(
                {
                    "condition": condition,
                    "window": w + 1,
                    "centre": (w + 0.5) * WINDOW,
                    **estimate(
                        win[:, :, w],
                        samples=options.samples,
                        rng=rng,
                        seeds=options.seeds,
                    ),
                }
            )
    readings = []
    me = cubes.get(options.method)
    if me is not None:
        rate = me[0]
        last = rate.shape[2] - 1
        e = estimate(
            rate[:, :, last] - rate[:, :, 0],
            samples=options.samples,
            rng=rng,
            seeds=options.seeds,
        )
        per = [e[f"seed_{s}"] for s in options.seeds]
        readings.append(
            {
                "reading": f"M1 {SHORT[options.method]} map 4 - map 1",
                **e,
                "holds": all(abs(v) <= DELTA for v in per),
            }
        )
        for other in (
            "full_context",
            "memo",
            "memo_fixed",
            "full_gru",
            ABLATION_CONDITION,
            "raw_segment",
        ):
            if cubes.get(other) is None or other == options.method:
                continue
            e = estimate(
                rate[:, :, last] - cubes[other][0][:, :, last],
                samples=options.samples,
                rng=rng,
                seeds=options.seeds,
            )
            per = [e[f"seed_{s}"] for s in options.seeds]
            tag = "M2 " if other in ("full_context", "memo") else ""
            readings.append(
                {
                    "reading": f"{tag}{SHORT[options.method]} - {SHORT[other]}, map 4",
                    **e,
                    "holds": e["lower"] > 0 and all(v >= DELTA for v in per),
                }
            )
    units = len(contract.roster(options.split)) * len(contract.evaluation.rollout_seeds)
    return {
        "present": present,
        "missing": missing,
        "per_map": per_map,
        "first_lap": first_lap,
        "windows": windows,
        "readings": readings,
        "units": units,
    }


def _change_marks(ax, options):
    for k in range(1, options.budget // options.period):
        ax.axvline(
            k * options.period,
            color=INK_FAINT,
            linewidth=0.7,
            linestyle=(0, (2, 2)),
            zorder=1,
        )


def draw_map_windows(ax, data, options):
    """(a) goals per 500-call window; dotted lines at the map changes."""
    top = 5.0
    for cell in data["present"]:
        mine = [r for r in data["windows"] if r["condition"] == cell.condition]
        x = [r["centre"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.4))
        top = max(top, 1.05 * max(r["upper"] for r in mine))
    ticks = list(range(0, options.budget + 1, options.period))
    ax.set_xlim(0, options.budget)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t:,}" for t in ticks])
    _change_marks(ax, options)
    ax.text(
        options.period + 0.01 * options.budget,
        0.97,
        "new map",
        transform=ax.get_xaxis_transform(),
        color=INK_MUTED,
        fontsize=8.0,
        va="top",
        ha="left",
    )
    ax.set_ylim(0, top)
    ax.set_xlabel("Charged calls")
    ax.set_ylabel("Goals per 500-call window")
    ax.set_title("(a) goals per window")
    style_axes(ax)


def draw_per_map(ax, data, options):
    """(b) goals per 500 calls on each map, the roster map first."""
    maps = options.budget // options.period
    top = 5.0
    for cell in data["present"]:
        mine = sorted(
            (r for r in data["per_map"] if r["condition"] == cell.condition),
            key=lambda r: r["map"],
        )
        x = [r["map"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.8))
        top = max(top, 1.05 * max(r["upper"] for r in mine))
    ax.set_xticks(range(1, maps + 1))
    ax.set_xticklabels(["1 (roster)", *(str(k) for k in range(2, maps + 1))])
    ax.set_xlim(0.7, maps + 0.3)
    ax.set_ylim(0, top)
    ax.set_xlabel("Map within the task")
    ax.set_ylabel("Goals per 500 calls on the map")
    ax.set_title("(b) goal rate per map")
    style_axes(ax)


def draw_first_lap(ax, data, options):
    """(c) calls the first finished lap on each map took (a lap that never
    finished is charged the whole budget): the speed of re-adaptation."""
    maps = options.budget // options.period
    for cell in data["present"]:
        mine = sorted(
            (r for r in data["first_lap"] if r["condition"] == cell.condition),
            key=lambda r: r["map"],
        )
        x = [r["map"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.8))
    ax.axhline(500, color=INK_FAINT, linewidth=0.7, linestyle=(0, (2, 2)), zorder=1)
    ax.text(0.75, 500, " timer", color=INK_MUTED, fontsize=8.0, va="bottom", ha="left")
    ax.set_xticks(range(1, maps + 1))
    ax.set_xticklabels(["1 (roster)", *(str(k) for k in range(2, maps + 1))])
    ax.set_xlim(0.7, maps + 0.3)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Map within the task")
    ax.set_ylabel("Calls for the first lap on the map")
    ax.set_title("(c) first lap on each map")
    style_axes(ax)


def map_caption(data, options):
    if not data["readings"]:
        return ""
    parts = [
        f"{r['reading']} {r['estimate']:+.1f} [{r['lower']:+.1f}, {r['upper']:+.1f}]"
        for r in data["readings"]
    ]
    head = (
        f"One task of {options.budget:,} charged calls; a new 15x15 maze and goal sequence at the first lap "
        f"boundary after every {options.period:,} calls, the action map kept, the memory carried "
        f"({data['units']} tasks, {len(options.seeds)} seeds, joint seed/task 95 % intervals). "
        "Goals per 500 calls on the map: " + "; ".join(parts) + "."
    )
    return "\n".join(textwrap.wrap(head, 150))


def build_map_change(layout, options=None):
    options = options or MapChangeOptions()
    sources = []
    data = compute_map_change(layout, options, sources)
    handles = legend_handles(data["present"]) + pending_handles(
        {"MazeRunner map change": data["missing"]},
        options.cells,
        reason="pending (no complete panels)",
    )
    title = (
        f"{ENVIRONMENTS['mazerunner'].title} · a new map of the trained size every {options.period:,} calls, frozen policies"
        + (
            ""
            if options.split == "confirmation"
            else f" ({options.split} roster, provisional)"
        )
    )
    with paper_style():
        fig = plt.figure(figsize=options.figsize, layout="constrained")
        axes = fig.subplots(1, 3)
        draw_map_windows(axes[0], data, options)
        draw_per_map(axes[1], data, options)
        draw_first_lap(axes[2], data, options)
        composite_legend(fig, handles, ncol=4, title=title)
        text = map_caption(data, options) if options.caption else ""
        if text:
            fig.text(
                0.5, -0.01, text, ha="center", va="top", fontsize=8.0, color=INK_MUTED
            )
    panels = {
        "goals_per_window": standalone(
            lambda ax: draw_map_windows(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
        "goal_rate_per_map": standalone(
            lambda ax: draw_per_map(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
        "first_lap_per_map": standalone(
            lambda ax: draw_first_lap(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
    }
    tables = {
        "per_map": data["per_map"],
        "first_lap": data["first_lap"],
        "windows": data["windows"],
        "readings": data["readings"],
    }
    notes = {
        "split": options.split,
        "budget": options.budget,
        "period": options.period,
        "present": [c.condition for c in data["present"]],
        "missing": [c.condition for c in data["missing"]],
        "units": data["units"],
        "method": options.method,
        "delta": DELTA,
        "declared": "before any panel",
        "checkpoint": "policy_epoch_999 (collection endpoint)",
        "interval": "95 % joint seed/task bootstrap, conditional on the trained seeds",
    }
    return BuiltFigure(options.name, fig, tables, notes, sources, panels)

In [ ]:
# Parameters: `split` the roster (confirmation, the paper's; development is
# the declared pre-read), `budget` the charged calls per map (4,000 declared),
# `cells` the memory carriers drawn, `samples` the bootstrap draws (2000).
options = LapsOptions(samples=2000, resampling_seed=0, split="confirmation")
laps = build_laps(layout, options)
display(laps.figure)

In [ ]:
# The roster, the cells present and pending, and each laps panel's acceptance
# check (its first lap reproduces the plain confirmation episode map for map).
print(
    {
        k: v
        for k, v in laps.notes.items()
        if k in ("split", "present", "missing", "lap_cleared_present", "prefix_checks")
    }
)

In [ ]:
# The last-window readings and the calls a lap takes, lap by lap.
display(Markdown(markdown_preview(laps.tables["readings"], limit=20)))
display(Markdown(markdown_preview(laps.tables["calls_per_lap"], limit=40)))

## A new map of the trained size inside the long task

The same 4,000-call task, but the maze and the goal sequence are replaced at the first lap boundary after every 1,000 calls, all at the trained 15 × 15 size. The hidden action permutation is kept, so what was learned about the action map stays valid while the walls and goals must be learned again; the memory is never reset and nothing in the packet marks the change. Panel (a) is the rate per 500-call window with the changes dotted, (b) the goal rate on each map, and (c) how many calls the first lap on each map takes — the speed of re-adaptation.


In [ ]:
# Parameters: `period` the calls between map changes (1,000 declared), `split`
# the roster, `samples` the bootstrap draws.
map_options = MapChangeOptions(samples=2000, resampling_seed=0, split="confirmation")
map_change = build_map_change(layout, map_options)
display(map_change.figure)
display(Markdown(markdown_preview(map_change.tables["readings"], limit=20)))

In [ ]:
# Save: both composites (PDF + PNG), every panel (PNG), the tables and the notes.
for built in (laps, map_change):
    for path in built.save(FIGURES_DIR, DATA_DIR):
        print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    built.close()